In [1]:
import math
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import QFTGate
from qiskit.quantum_info import Operator

# ── Build QFT matrix ──────────────────────────────────────
qc = QuantumCircuit(4)
qc.append(QFTGate(4), range(4))
U = np.array(Operator(qc).data)  # 16x16 complex matrix

# ── Butterfly on a single row/column (uses magnitudes) ────
def butterfly_collapse(values):
    current = [abs(float(v.real)) + abs(float(v.imag)) for v in values]
    while len(current) > 1:
        next_vals = []
        for i in range(0, len(current), 2):
            j, k = current[i], current[i+1]
            next_vals.append(math.sqrt(j**2 + k**2))
        current = next_vals
    return current[0]

# ── One full pass: butterfly every row, then every column ─
def butterfly_pass(matrix):
    n = matrix.shape[0]
    # Row pass — each row collapses to 1 number → becomes diagonal
    row_vals = np.array([butterfly_collapse(matrix[i]) for i in range(n)])
    # Column pass — each col collapses to 1 number
    col_vals = np.array([butterfly_collapse(matrix[:, j]) for j in range(n)])
    # Build new matrix: diagonal from row*col products, zeros elsewhere
    new_matrix = np.zeros((n, n))
    for i in range(n):
        new_matrix[i, i] = row_vals[i] * col_vals[i]
    return new_matrix, row_vals, col_vals

# ── Run multiple passes ────────────────────────────────────
def run_diagonal_collapse(matrix, passes=3):
    current = matrix.copy()
    for p in range(passes):
        print(f"\n{'='*50}")
        print(f"  PASS {p+1}")
        print(f"{'='*50}")
        current, row_vals, col_vals = butterfly_pass(current)
        print(f"\n  Row values:  {np.round(row_vals, 4)}")
        print(f"  Col values:  {np.round(col_vals, 4)}")
        print(f"\n  Diagonal:\n  {np.round(np.diag(current), 4)}")
        off_diag = current - np.diag(np.diag(current))
        print(f"  Off-diagonal sum: {np.sum(np.abs(off_diag)):.6f}")
    return current

result = run_diagonal_collapse(U, passes=3)


  PASS 1

  Row values:  [1.     1.2663 1.2247 1.2663 1.     1.2663 1.2247 1.2663 1.     1.2663
 1.2247 1.2663 1.     1.2663 1.2247 1.2663]
  Col values:  [1.     1.2663 1.2247 1.2663 1.     1.2663 1.2247 1.2663 1.     1.2663
 1.2247 1.2663 1.     1.2663 1.2247 1.2663]

  Diagonal:
  [1.     1.6036 1.5    1.6036 1.     1.6036 1.5    1.6036 1.     1.6036
 1.5    1.6036 1.     1.6036 1.5    1.6036]
  Off-diagonal sum: 0.000000

  PASS 2

  Row values:  [1.     1.6036 1.5    1.6036 1.     1.6036 1.5    1.6036 1.     1.6036
 1.5    1.6036 1.     1.6036 1.5    1.6036]
  Col values:  [1.     1.6036 1.5    1.6036 1.     1.6036 1.5    1.6036 1.     1.6036
 1.5    1.6036 1.     1.6036 1.5    1.6036]

  Diagonal:
  [1.     2.5714 2.25   2.5714 1.     2.5714 2.25   2.5714 1.     2.5714
 2.25   2.5714 1.     2.5714 2.25   2.5714]
  Off-diagonal sum: 0.000000

  PASS 3

  Row values:  [1.     2.5714 2.25   2.5714 1.     2.5714 2.25   2.5714 1.     2.5714
 2.25   2.5714 1.     2.5714 2.25   2.5714]

In [2]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector, Parameter
from qiskit.circuit.library import RYGate, QFTGate
from qiskit.quantum_info import Statevector
from medmnist import BreastMNIST
from skimage.transform import resize

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

# ============================================================
# 1) DATA
# ============================================================
def load_breast_data(target_size=(4,4)):
    dataset = BreastMNIST(split='train', download=True)
    X, y = [], []
    for i in range(len(dataset)):
        img_pil, label = dataset[i]
        img = np.array(img_pil)
        img_small = resize(img, target_size, anti_aliasing=True).flatten()
        X.append(img_small)
        y.append(label)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64).flatten()

X, y = load_breast_data()
print(f"Loaded {X.shape[0]} samples, each with {X.shape[1]} pixels.")
print("Class counts:", np.bincount(y))

# scale to [0, pi]
X = X / (X.max() + 1e-12) * np.pi

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ============================================================
# 2) PARAMETERIZED CIRCUIT TEMPLATE (pixels + trainable qsiht)
# ============================================================
num_qubits = 4

pixel_params = [Parameter(f"x{i}") for i in range(16)]
theta_params = ParameterVector("θ", 15)

qc_template = QuantumCircuit(num_qubits)

# angle-encode pixels: 4 layers × 4 qubits = 16 params
for layer in range(4):
    for q in range(4):
        idx = layer * 4 + q
        qc_template.ry(pixel_params[idx], q)

# QsiHT block (15 controlled RY)
sub_qc = QuantumCircuit(num_qubits, name="QsiHT")

def add_cry(qc, theta, controls, ctrl_state, target):
    gate = RYGate(theta).control(len(controls), ctrl_state=ctrl_state)
    qc.append(gate, controls + [target])

add_cry(sub_qc, theta_params[0],  [0,1,2], "000", 3)
add_cry(sub_qc, theta_params[1],  [0,1,2], "100", 3)
add_cry(sub_qc, theta_params[2],  [0,1,3], "000", 2)
add_cry(sub_qc, theta_params[3],  [0,1,2], "010", 3)
add_cry(sub_qc, theta_params[4],  [0,1,2], "110", 3)
add_cry(sub_qc, theta_params[5],  [0,1,3], "010", 2)
add_cry(sub_qc, theta_params[6],  [0,2,3], "000", 1)
add_cry(sub_qc, theta_params[7],  [0,1,2], "001", 3)
add_cry(sub_qc, theta_params[8],  [0,1,2], "101", 3)
add_cry(sub_qc, theta_params[9],  [0,1,3], "001", 2)
add_cry(sub_qc, theta_params[10], [0,1,2], "011", 3)
add_cry(sub_qc, theta_params[11], [0,1,2], "111", 3)
add_cry(sub_qc, theta_params[12], [0,1,3], "011", 2)
add_cry(sub_qc, theta_params[13], [0,2,3], "001", 1)
add_cry(sub_qc, theta_params[14], [1,2,3], "000", 0)

qc_template.append(sub_qc.to_gate(), range(num_qubits))

# Optional: 2D QFT (rows + cols)
USE_QFT = True
if USE_QFT:
    qc_template.append(QFTGate(2), [0,1])  # rows
    qc_template.append(QFTGate(2), [2,3])  # cols

# ============================================================
# 3) FEATURE EXTRACTOR: exact probs from statevector
# ============================================================
def probs_from_params(x16, theta15):
    bind = {pixel_params[i]: float(x16[i]) for i in range(16)}
    bind.update({theta_params[j]: float(theta15[j]) for j in range(15)})
    bound = qc_template.assign_parameters(bind, inplace=False)

    sv = Statevector.from_instruction(bound)
    probs = np.abs(sv.data) ** 2
    return probs.astype(np.float32)  # shape (16,)

# Pick initial θ (you can later tune these with classical optimization)
theta_init = np.random.uniform(-np.pi, np.pi, size=15).astype(np.float32)

def build_features(Xdata, theta15):
    feats = np.zeros((len(Xdata), 16), dtype=np.float32)
    for i in range(len(Xdata)):
        feats[i] = probs_from_params(Xdata[i], theta15)
        if (i+1) % 200 == 0:
            print(f"features: {i+1}/{len(Xdata)}")
    return feats

print("\nBuilding quantum features (train)...")
F_train = build_features(X_train, theta_init)
print("Building quantum features (test)...")
F_test  = build_features(X_test, theta_init)

# ============================================================
# 4) TRAIN CLASSIFIER for featurs
# ============================================================
X_train_t = torch.tensor(F_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t  = torch.tensor(F_test, dtype=torch.float32)
y_test_t  = torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=32)

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 2)
        )
    def forward(self, x):
        return self.fc(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Net().to(device)

# class weights (imbalance)
counts = np.bincount(y_train)
w = (1.0 / counts)
w = w / w.sum() * len(counts)
criterion = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32).to(device))

optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

for epoch in range(30):
    model.train()
    total_loss = 0.0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        out = model(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # eval
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for bx, by in test_loader:
            bx, by = bx.to(device), by.to(device)
            pred = model(bx).argmax(dim=1)
            correct += (pred == by).sum().item()
            total += by.size(0)

    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1:02d} | loss={total_loss/len(train_loader):.4f} | test_acc={correct/total:.4f}")

print("\nDone.")

Loaded 546 samples, each with 16 pixels.
Class counts: [147 399]

Building quantum features (train)...


KeyboardInterrupt: 